<a href="https://colab.research.google.com/github/garam827/LLM_Study/blob/main/GEMINI_API_AI_%ED%88%AC%EC%9E%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 펑션 콜링

In [2]:
from datetime import datetime
import pytz

def get_current_time(timezone: str = 'Asia/Seoul'):
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    now_timezone = f'{now} {timezone}'
    print(now_timezone)
    print(now)
    return now_timezone


tools = [
    {
        'type' : 'function',
        'function' : {
            'name' : 'get_current_time',
            'description' : '해당 타임존의 날짜와 시간을 반환합니다.',
            'parameters' : {
                'type' : 'object',
                'properties' : {
                    'timezone' : {
                        'type' : 'string',
                        'description' : '현재 날짜와 시간을 반환할 타임존을 입력하세요.(예: Asia/Seoul)'
                    }
                },
                'required' : ['timezone'],
            }
        }
    }
]

In [3]:
get_current_time()

2026-03-24 22:13:39 Asia/Seoul
2026-03-24 22:13:39


'2026-03-24 22:13:39 Asia/Seoul'

In [4]:
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')

In [48]:
from openai import OpenAI

import json

client = OpenAI(
    api_key     = api_key,
    base_url    = "https://generativelanguage.googleapis.com/v1beta/openai/"
    # base_url    = "https://generativelanguage.googleapis.com"
)

#  https://ai.google.dev/gemini-api/docs/models?hl=ko#previous_models

def get_ai_response(messages, tools = None):
    response = client.chat.completions.create(
        # model       = "gemini-3-flash-preview",
        model       = "gemini-2.5-flash-lite",
        # model       = "gemini-3.1-flash-lite",
        # model       = "gemma-3-27b-it",
        # model       = 'gemini-1.5-flash',
        # model       = 'gemini-1.5-pro-latest',
        messages    = messages,
        tools       = tools,
    )

    return response

In [40]:
# 초기 시스템 메시지 설정
messages = [
    {"role": "system",  "content": "너는 사용자를 도와주는 상담사야."},
]

while True:
    user_input  = input("사용자: ")

    if user_input == "exit":
        break

    messages.append({"role": "user",  "content": user_input})

    ai_response = get_ai_response(messages, tools = tools)
    ai_message = ai_response.choices[0].message
    print(ai_message)

    tool_calls = ai_message.tool_calls
    if tool_calls:
        tool_name = tool_calls[0].function.name
        tool_call_id = tool_calls[0].id

        arguments = json.loads(ai_message.tool_calls[0].function.arguments)

        if tool_name == "get_current_time":
            messages.append({
                "role": "function",
                "tool_call_id": tool_call_id,
                "name" : tool_name,
                "content": get_current_time(timezone = arguments['timezone'])
            })

        ai_response = get_ai_response(messages, tools = tools)
        ai_message = ai_response.choices[0].message

    messages.append(ai_message)

    print("AI\t: " + ai_message.content)

사용자: 안녕
ChatCompletionMessage(content='안녕하세요! 무엇을 도와드릴까요?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)
AI	: 안녕하세요! 무엇을 도와드릴까요?
사용자: 지금 몇시야??
ChatCompletionMessage(content='어느 도시의 현재 시간을 알려드릴까요? 도시 이름을 알려주시면 정확한 시간을 찾아드릴게요.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)
AI	: 어느 도시의 현재 시간을 알려드릴까요? 도시 이름을 알려주시면 정확한 시간을 찾아드릴게요.
사용자: 한국
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='function-call-15292031713229253588', function=Function(arguments='{"timezone":"Asia/Seoul"}', name='get_current_time'), type='function')])
2026-03-24 22:52:04 Asia/Seoul
2026-03-24 22:52:04
AI	: 현재 한국 시간은 2026년 3월 24일 오후 10시 52분입니다.
사용자: exit


## 여러 도시 시간 한번에 대답

In [49]:
# 초기 시스템 메시지 설정
messages = [
    {"role": "system",  "content": "너는 사용자를 도와주는 상담사야."},
]

while True:
    user_input  = input("사용자: ")

    if user_input == "exit":
        break

    messages.append({"role": "user",  "content": user_input})

    ai_response = get_ai_response(messages, tools = tools)
    ai_message = ai_response.choices[0].message
    print(ai_message)

    tool_calls = ai_message.tool_calls
    if tool_calls:
        for tool_call in tool_calls:
            tool_name = tool_call.function.name
            tool_call_id = tool_call.id

            arguments = json.loads(tool_call.function.arguments)

            if tool_name == "get_current_time":
                messages.append({
                    "role": "function",
                    "tool_call_id": tool_call_id,
                    "name" : tool_name,
                    "content": get_current_time(timezone = arguments['timezone'])
                })
        messages.append({"role" : "system", "content" : "이제 주어진 결과를 바탕으로 답변할 차례다."})

        ai_response = get_ai_response(messages, tools = tools)
        ai_message = ai_response.choices[0].message

    messages.append(ai_message)

    print("AI\t: " + ai_message.content)

사용자: 중국, 영국, 프랑스 시간 알려줘.
ChatCompletionMessage(content='어느 도시의 시간을 알려드릴까요?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)
AI	: 어느 도시의 시간을 알려드릴까요?
사용자: 베이징, 런던, 파리
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='function-call-5891134149492554109', function=Function(arguments='{"timezone":"Asia/Shanghai"}', name='get_current_time'), type='function'), ChatCompletionMessageFunctionToolCall(id='function-call-5891134149492554920', function=Function(arguments='{"timezone":"Europe/London"}', name='get_current_time'), type='function'), ChatCompletionMessageFunctionToolCall(id='function-call-5891134149492555731', function=Function(arguments='{"timezone":"Europe/Paris"}', name='get_current_time'), type='function')])
2026-03-24 22:41:15 Asia/Shanghai
2026-03-24 22:41:15
2026-03-24 14:41:15 Europe/London
2026-03-24 14:41:15